# 07. Sensitivity robustness checks

This notebook extends the primary results (notebook 06) to the full κ × π grid (28 scenarios across 4 base rates: π_Primary ∈ {0.05, 0.10, 0.20, 0.50}). It examines:

1. **Rater-model calibration**: realised κ vs target κ across the full grid (loaded from `outputs/sensitivity/tables/rater_model_verification.csv`).
2. **Analytical ceiling effect**: at the lowest base rate (π = 0.05) and highest target (κ = 0.80), the rater model cannot reach the target — the analytical maximum κ at π = 0.05 with the 2% Uncertain rate is approximately 0.71. The simulation should converge to this ceiling at that scenario cell.
3. **Outcome variation across base rates**: the four governance outcomes vary not only with κ but also with π. The full-grid table is the basis of the supplementary figure (`outputs/sensitivity/figures/figure_supp_kappa_pi_grid.png`).

These checks verify that the simulation behaves correctly under the modelling assumptions stated in [`docs/sensitivity/modelling_assumptions.md`](../docs/sensitivity/modelling_assumptions.md). They are robustness checks against the rater model's internal consistency, not against external data.

**Epistemic Notice — Sensitivity Analysis Notebooks**

Unlike Paper 2's earlier notebooks (01–04), which render structured data extracted from the manuscript without generating new scientific claims, the notebooks in this sensitivity series (05–08) DO produce new computational findings under stated modelling assumptions. Specifically, they explore how variation in inter-rater agreement (κ) on the Negative Harm Test (NHT) propagates to four governance outcomes, calibrating κ tolerance regimes for the tier-classification rule.

This work is consistent with the Threshold Justification Stack's non-compensatory architecture. Each governance gate has its own threshold expressed in its own evidential "currency"; the framework rejects exchange rates between currencies. The κ-sensitivity simulation operates entirely within the NHT's own currency — it varies what the tier-classification reliability threshold should be, given the rule's purpose. It does not propose that κ performance on the NHT could compensate for thresholds in other gates of the framework.

Findings throughout this series are conditional on the modelling assumptions in [`docs/sensitivity/modelling_assumptions.md`](../docs/sensitivity/modelling_assumptions.md). Readers should treat results as methodological calibration evidence — informing what tier-classification reliability would need to look like for the NHT to function as designed — not as standalone empirical claims independent of the framework.

In [1]:
# Bootstrap: walk upward to find repo root, add to sys.path, then import tjs_sensitivity.*
import sys
from pathlib import Path

for _cand in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_cand / "config" / "harness_settings.json").is_file():
        _s = str(_cand)
        if _s not in sys.path:
            sys.path.insert(0, _s)
        break

from tjs_sensitivity.bootstrap import prepare_notebook
from tjs_sensitivity.bootstrap import (
    SENSITIVITY_TABLES_DIR,
    SENSITIVITY_INPUTS_DIR,
)

REPO_ROOT = prepare_notebook()
print(f"Repo root: {REPO_ROOT}")

Repo root: /workspace


## 07.1 Rater-model calibration: realised κ vs target κ

The rater model (`tjs_sensitivity.rater_model`) calibrates per-rater error rates by numerical inversion to achieve the target κ on the binary-collapsed classifications. Across the κ × π grid, the **realised κ** (mean across replicates) should match the **target κ** within tight tolerance — modulo the analytical ceiling at low base rates (covered in §07.2).

The simulation persists realised vs target κ for every scenario in `outputs/sensitivity/tables/rater_model_verification.csv`. **The κ-sensitivity simulation is conditional on stated modelling assumptions** (P3-C51): a probabilistic rater-error model parameterised to target κ, a binary-plus-Uncertain output alphabet, default-to-Primary adjudication, and a base-rate prior on π_Primary. The calibration check below validates that the rater-error model assumption is honestly applied — the model achieves what it claims to.

In [2]:
# Load the rater-model verification table
import csv

verif_path = REPO_ROOT / SENSITIVITY_TABLES_DIR / "rater_model_verification.csv"
with open(verif_path) as f:
    verif_rows = list(csv.DictReader(f))

print(f"Loaded {len(verif_rows)} verification rows (28 scenarios = 7 κ × 4 π)")
print()
print(f"{'sc_id':>5} {'π':>6} {'target κ':>10} {'realised κ':>14} {'abs_diff':>14}")
for r in verif_rows:
    print(
        f"{int(r['scenario_id']):>5} "
        f"{float(r['base_rate_primary']):>6.2f} "
        f"{float(r['target_kappa']):>10.2f} "
        f"{float(r['realised_kappa_mean']):>14.5f} "
        f"{float(r['abs_diff']):>14.5f}"
    )

Loaded 28 verification rows (28 scenarios = 7 κ × 4 π)

sc_id      π   target κ     realised κ       abs_diff
    0   0.05       0.20        0.19675        0.00325
    1   0.05       0.30        0.29680        0.00320
    2   0.05       0.40        0.39751        0.00249
    3   0.05       0.50        0.49757        0.00243
    4   0.05       0.60        0.60565        0.00565
    5   0.05       0.70        0.70171        0.00171
    6   0.05       0.80        0.71243        0.08757
    7   0.10       0.20        0.20127        0.00127
    8   0.10       0.30        0.29765        0.00235
    9   0.10       0.40        0.39911        0.00089
   10   0.10       0.50        0.49929        0.00071
   11   0.10       0.60        0.59804        0.00196
   12   0.10       0.70        0.70118        0.00118
   13   0.10       0.80        0.79700        0.00300
   14   0.20       0.20        0.20530        0.00530
   15   0.20       0.30        0.29788        0.00212
   16   0.20       0.40   

## 07.2 The analytical ceiling at low base rates

At the lowest base rate (π = 0.05), the rater model has an **analytical maximum κ** below 1.0 because of the Uncertain-rate floor (u = 0.02). Specifically: even with zero classification error, two raters who disagree on Uncertain-events introduce non-zero apparent disagreement, which caps the achievable Cohen's κ.

The modelling-assumptions document (`docs/sensitivity/modelling_assumptions.md` §2) states the analytical maximum κ at π = 0.05 with u = 0.02 is approximately 0.71. Therefore, when the κ grid asks for target κ = 0.80 at π = 0.05, the simulation cannot reach the target — it converges to the analytical ceiling. This is a **documented mathematical property of the rater model** (P3-C51, simulation-specific limits) — the joint-parameter constraint between base rate, Uncertain rate, and achievable κ.

This is **not a bug**. The simulation correctly identifies the ceiling rather than masking it. The ceiling assert below verifies that the simulation behaves as the rater model's mathematics predicts, not as the target κ would suggest if the ceiling were ignored.

In [3]:
# Locate the ceiling scenario: (π=0.05, target κ=0.80)
ceiling_row = next(
    r for r in verif_rows
    if float(r['base_rate_primary']) == 0.05 and float(r['target_kappa']) == 0.80
)

target = float(ceiling_row['target_kappa'])
realised = float(ceiling_row['realised_kappa_mean'])
abs_diff = float(ceiling_row['abs_diff'])

print(f"Ceiling scenario (scenario_id={ceiling_row['scenario_id']}):")
print(f"  π = 0.05, target κ = {target:.2f}")
print(f"  realised κ (mean across 100 replicates) = {realised:.4f}")
print(f"  |realised − target| = {abs_diff:.4f}")
print(f"  modelling-assumptions doc states analytical maximum κ at π=0.05, u=0.02 ≈ 0.71")
print()

# Compare ceiling-scenario abs_diff to other scenarios at π=0.05
print("All π=0.05 scenarios:")
print(f"{'target κ':>10} {'realised κ':>14} {'abs_diff':>14}")
for r in verif_rows:
    if float(r['base_rate_primary']) == 0.05:
        print(
            f"{float(r['target_kappa']):>10.2f} "
            f"{float(r['realised_kappa_mean']):>14.5f} "
            f"{float(r['abs_diff']):>14.5f}"
        )

Ceiling scenario (scenario_id=6):
  π = 0.05, target κ = 0.80
  realised κ (mean across 100 replicates) = 0.7124
  |realised − target| = 0.0876
  modelling-assumptions doc states analytical maximum κ at π=0.05, u=0.02 ≈ 0.71

All π=0.05 scenarios:
  target κ     realised κ       abs_diff
      0.20        0.19675        0.00325
      0.30        0.29680        0.00320
      0.40        0.39751        0.00249
      0.50        0.49757        0.00243
      0.60        0.60565        0.00565
      0.70        0.70171        0.00171
      0.80        0.71243        0.08757


## 07.3 Outcome variation across base rates

The full κ × π grid (`outputs/sensitivity/tables/sensitivity_full_grid.csv`) contains all 28 scenarios. Outcomes vary not only with target κ but also with base rate π — at any fixed κ, the four governance outcomes shift across the four base-rate scenarios. This grid is the basis of the supplementary figure (`outputs/sensitivity/figures/figure_supp_kappa_pi_grid.png`) and is part of the **simulation-specific limits** (P3-C51) the manuscript discloses: the base-rate prior is a modelling input, and outcomes depend on it.

For the unsafe-Secondary rate at fixed κ = 0.60: at π = 0.05 the rate is 0.0% (no latent-Primary thresholds to misclassify); at π = 0.10 it is 0.07%; at π = 0.20 it is in between; at π = 0.50 it is materially higher (more latent-Primary thresholds to misclassify, all else equal). The variation pattern is structural — driven by base-rate composition, not by the rater model's calibration behaviour. The assert below verifies this base-rate-driven variation exists in the full grid.

In [4]:
# Load the full κ × π grid; show unsafe-Secondary rate variation across π at fixed κ=0.60
full_grid_path = REPO_ROOT / SENSITIVITY_TABLES_DIR / "sensitivity_full_grid.csv"
with open(full_grid_path) as f:
    full_rows = list(csv.DictReader(f))

print(f"Loaded {len(full_rows)} scenarios from full grid")
print()

# Filter to κ=0.60
k60_rows = [r for r in full_rows if float(r['target_kappa']) == 0.60]
print(f"At fixed target κ = 0.60, unsafe-Secondary rate varies across base rates:")
print(f"{'π':>6} {'unsafe-Sec rate':>16} {'MCSE':>10}")
for r in sorted(k60_rows, key=lambda x: float(x['base_rate_primary'])):
    print(
        f"{float(r['base_rate_primary']):>6.2f} "
        f"{float(r['unsafe_secondary_rate_mean']):>16.5f} "
        f"{float(r['unsafe_secondary_rate_mcse']):>10.5f}"
    )

Loaded 28 scenarios from full grid

At fixed target κ = 0.60, unsafe-Secondary rate varies across base rates:
     π  unsafe-Sec rate       MCSE
  0.05          0.00000    0.00000
  0.10          0.00070    0.00026
  0.20          0.00469    0.00043
  0.50          0.01132    0.00047


## 07.4 Pass criteria

The asserts below verify three experiment-internal properties of the simulation. Per the WS-2.6 Phase 8.1 status taxonomy + Phase 8.3 experiment-authoritative discipline (`feedback_experiment_authoritative_manuscript_downstream.md`):

- **assert 1**: Calibration tolerance — across all 27 non-ceiling scenarios, abs_diff between realised κ and target κ is below 0.015. **Tolerance source**: empirical-magnitude observation. Inspecting `rater_model_verification.csv`, the maximum non-ceiling abs_diff observed across the grid is ≈ 0.01 (at the [π=0.50, target κ=0.20] cell). The 0.015 tolerance reflects floating-point precision of the rater-error-model's numerical inversion plus modest empirical headroom (~50%) over the observed maximum. The assert verifies the simulation's actual numerical behaviour, not a manuscript-stated value. (**P3-C51** — VERIFIED on pass; the rater-error-model modelling-assumption is honestly applied within the model's numerical-inversion precision.)

- **assert 2**: Ceiling effect existence — at (π=0.05, target κ=0.80), the realised κ is materially below target (abs_diff > 0.05), AND realised κ falls in [0.70, 0.72]. **Bound source**: the analytical ceiling derives from the rater model's closed-form constraint (u=0.02 floor + π=0.05 base rate). The bound [0.70, 0.72] is a tight band around the analytical-ceiling value (~0.71 per `docs/sensitivity/modelling_assumptions.md` §2). The assert verifies the simulation converges to the analytical maximum κ where the target exceeds achievability. (**P3-C51** — VERIFIED on pass; the modelling-assumption analytical limit holds in the simulation.)

- **assert 3**: Outcome variation across base rates — at fixed target κ=0.60, the unsafe-Secondary rate varies across the four base rates (range ≥ 0.005). **Threshold source**: empirical observation that the range across the four π values at κ=0.60 is meaningful (≥ 0.005), distinguishing the full grid from a κ-only sweep. The assert verifies the structural property that the simulation's base-rate dimension produces material outcome variation. (No specific claim ID anchor; the assert demonstrates a property of the full grid's design.)

Claim **P3-C49** (threshold coupling Layer 4 inadequately addressed by current governance standards) earns **Traced** status with **no anchor in this notebook** (Walter Phase 8.4 discipline). The claim is a manuscript-discussion assertion about other governance standards; the simulation has no access to other governance standards and cannot substantiate or refute the claim. The substantive content of P3-C49 lives in the manuscript's Discussion of TJS Layer 4 — a manuscript-only claim that does not derive from the sensitivity simulation.

Claim **P3-C52** (L4 sensitivity / threshold coupling least methodologically mature) earns **Traced** status: narrative claim about methodological maturity, not a property the simulation can verify.

In [5]:
# Experiment-anchored pass-criterion asserts

# === assert 1: calibration tolerance across non-ceiling scenarios (P3-C51) ===
# All scenarios EXCEPT (π=0.05, κ=0.80) should have abs_diff < tolerance.
CALIBRATION_TOLERANCE = 0.015
calibration_violations = []
for r in verif_rows:
    pi = float(r['base_rate_primary'])
    k = float(r['target_kappa'])
    abs_diff_v = float(r['abs_diff'])
    is_ceiling = (pi == 0.05 and k == 0.80)
    if not is_ceiling and abs_diff_v >= CALIBRATION_TOLERANCE:
        calibration_violations.append((pi, k, abs_diff_v))
assert not calibration_violations, \
    f"Calibration violations: {calibration_violations}"
print(f"assert 1 (P3-C51 calibration tolerance < {CALIBRATION_TOLERANCE} for all 27 non-ceiling scenarios) ✓")

# === assert 2: ceiling effect at (π=0.05, κ=0.80) (P3-C51) ===
# Two-part check: abs_diff materially large AND realised κ in analytical-ceiling band [0.70, 0.72]
CEILING_MIN_DIFF = 0.05  # ceiling effect is material, not just MCSE
CEILING_BAND = (0.70, 0.72)  # analytical-ceiling band per docs/sensitivity/modelling_assumptions.md §2

assert abs_diff > CEILING_MIN_DIFF, \
    f"Ceiling effect not material at (π=0.05, κ=0.80): abs_diff={abs_diff:.4f} <= {CEILING_MIN_DIFF}"
assert CEILING_BAND[0] <= realised <= CEILING_BAND[1], \
    f"Ceiling realised κ outside analytical band: realised={realised:.4f}, band={CEILING_BAND}"
print(f"assert 2 (P3-C51 ceiling effect: realised κ={realised:.4f} in {CEILING_BAND}, abs_diff={abs_diff:.4f}>{CEILING_MIN_DIFF}) ✓")

# === assert 3: outcome variation across base rates at fixed κ=0.60 (no specific claim ID anchor) ===
# Per Walter Phase 8.4: variation across base rates is a structural property of the full grid;
# it does not anchor to P3-C49 (which is a meta-claim about other governance standards).
k60_unsafe_rates = sorted(float(r['unsafe_secondary_rate_mean']) for r in k60_rows)
variation_range = k60_unsafe_rates[-1] - k60_unsafe_rates[0]
MIN_VARIATION = 0.005

assert variation_range >= MIN_VARIATION, \
    f"Insufficient base-rate variation at κ=0.60: range={variation_range:.5f} < {MIN_VARIATION}"
print(f"assert 3 (full-grid base-rate variation at κ=0.60: range={variation_range:.5f} >= {MIN_VARIATION}; "
      f"min={k60_unsafe_rates[0]:.5f}, max={k60_unsafe_rates[-1]:.5f}) ✓")

print()
print("ALL THREE PASS-CRITERION ASSERTS HOLD")
print("Per Phase 8.1 status taxonomy + Phase 8.3/8.4 experiment-authoritative discipline:")
print("  P3-C49 (threshold coupling Layer 4 inadequately addressed)  → Traced (NO anchor)")
print("    (Per Walter Phase 8.4: claim is a meta-statement about other governance")
print("     standards; the simulation cannot substantiate or refute it. Substantive")
print("     content of C49 lives in the manuscript Discussion, not this notebook.)")
print("  P3-C51 (simulation-specific limits)                         → VERIFIED")
print("    (calibration + ceiling asserts validate the rater-error model assumption)")
print("  P3-C52 (L4 sensitivity / threshold coupling least mature)   → Traced")
print("    (narrative claim about methodological maturity)")

assert 1 (P3-C51 calibration tolerance < 0.015 for all 27 non-ceiling scenarios) ✓
assert 2 (P3-C51 ceiling effect: realised κ=0.7124 in (0.7, 0.72), abs_diff=0.0876>0.05) ✓
assert 3 (full-grid base-rate variation at κ=0.60: range=0.01132 >= 0.005; min=0.00000, max=0.01132) ✓

ALL THREE PASS-CRITERION ASSERTS HOLD
Per Phase 8.1 status taxonomy + Phase 8.3/8.4 experiment-authoritative discipline:
  P3-C49 (threshold coupling Layer 4 inadequately addressed)  → Traced (NO anchor)
    (Per Walter Phase 8.4: claim is a meta-statement about other governance
     standards; the simulation cannot substantiate or refute it. Substantive
     content of C49 lives in the manuscript Discussion, not this notebook.)
  P3-C51 (simulation-specific limits)                         → VERIFIED
    (calibration + ceiling asserts validate the rater-error model assumption)
  P3-C52 (L4 sensitivity / threshold coupling least mature)   → Traced
    (narrative claim about methodological maturity)


---

**Notebook 07 complete.** Robustness checks verify experiment-internal properties: the rater-model calibration is tight across the 27 non-ceiling scenarios; the ceiling effect at (π=0.05, target κ=0.80) appears where the rater model's mathematics predicts; outcome variation across base rates at fixed κ=0.60 is structurally present.

Per the WS-2.6 Phase 8.3 experiment-authoritative discipline, these asserts verify properties of the simulation itself rather than agreement with manuscript-stated ranges.

The next notebook (`08_sensitivity_discussion_and_implications.ipynb`) reconnects the sensitivity findings to the manuscript's Discussion sections — falsification conditions, proposed pilot design, integration mechanics, and anti-gaming considerations. Most claims in notebook 08 are narrative (Traced status); one or two have computational handles for VERIFIED status.